# 02 – Model Training

This notebook covers the complete model training workflow:
1. Data loading and preprocessing
2. Train/test split with stratification
3. Feature scaling with StandardScaler
4. Logistic Regression training (balanced class weights)
5. Evaluation: classification report, confusion matrix, ROC-AUC, Precision-Recall
6. Save model and scaler to `models/` using joblib
7. Test model loading and inference

In [ ]:
import os
import sys

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    PrecisionRecallDisplay,
    RocCurveDisplay,
    classification_report,
    confusion_matrix,
    f1_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

sys.path.insert(0, os.path.join('..', 'data'))
from generate_synthetic_data import generate_synthetic_data

%matplotlib inline
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

## 1. Load Data

In [ ]:
DATA_PATH  = os.path.join('..', 'data', 'creditcard.csv')
MODEL_DIR  = os.path.join('..', 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

if not os.path.exists(DATA_PATH):
    print('Generating dataset…')
    df = generate_synthetic_data(n_samples=100_000, fraud_ratio=0.02)
    df.to_csv(DATA_PATH, index=False)

df = pd.read_csv(DATA_PATH)
print(f'Shape: {df.shape}')
print(f'Fraud ratio: {df["Class"].mean() * 100:.2f}%')
df.head()

## 2. Preprocessing and Train/Test Split (80/20 stratified)

In [ ]:
feature_cols = [c for c in df.columns if c != 'Class']
X = df[feature_cols].values
y = df['Class'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f'Train size: {len(X_train):,}  |  Test size: {len(X_test):,}')
print(f'Fraud in train: {y_train.sum()} ({y_train.mean()*100:.2f}%)')
print(f'Fraud in test : {y_test.sum()} ({y_test.mean()*100:.2f}%)')

## 3. Train Logistic Regression

In [ ]:
model = LogisticRegression(
    class_weight='balanced', max_iter=1000, random_state=42, solver='lbfgs'
)
model.fit(X_train_scaled, y_train)
print('Training complete.')

## 4. Evaluation

In [ ]:
y_pred  = model.predict(X_test_scaled)
y_proba = model.predict_proba(X_test_scaled)[:, 1]

print('Classification Report:')
print(classification_report(y_test, y_pred, target_names=['Normal', 'Fraud']))

roc_auc = roc_auc_score(y_test, y_proba)
f1      = f1_score(y_test, y_pred)
print(f'ROC-AUC : {roc_auc:.4f}')
print(f'F1 Score: {f1:.4f}')

### Confusion Matrix

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Normal', 'Fraud'])
disp.plot(cmap='Blues', ax=ax)
ax.set_title('Confusion Matrix')
plt.show()

### ROC Curve

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
RocCurveDisplay.from_predictions(y_test, y_proba, ax=ax)
ax.plot([0, 1], [0, 1], 'k--', label='Random classifier')
ax.set_title(f'ROC Curve  (AUC = {roc_auc:.4f})')
ax.legend()
plt.show()

### Precision-Recall Curve

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
PrecisionRecallDisplay.from_predictions(y_test, y_proba, ax=ax)
ax.set_title('Precision-Recall Curve')
plt.show()

## 5. Save Model and Scaler

In [ ]:
model_path  = os.path.join(MODEL_DIR, 'fraud_model.pkl')
scaler_path = os.path.join(MODEL_DIR, 'scaler.pkl')

joblib.dump(model,  model_path)
joblib.dump(scaler, scaler_path)

print(f'Model  saved → {model_path}')
print(f'Scaler saved → {scaler_path}')

## 6. Verify – Load and Run Inference

In [ ]:
loaded_model  = joblib.load(model_path)
loaded_scaler = joblib.load(scaler_path)

sample = X_test[:5]
sample_scaled = loaded_scaler.transform(sample)
preds  = loaded_model.predict(sample_scaled)
probas = loaded_model.predict_proba(sample_scaled)[:, 1]

for i, (p, prob) in enumerate(zip(preds, probas)):
    actual = y_test[i]
    print(f'Sample {i+1}: actual={actual}  pred={p}  prob={prob:.4f}')